In [ ]:

# ETAPA 4 - PASSO 1
# Validar previsões: FastAPI versus Notebook
# .\.venv\Scripts\python.exe -m uvicorn app.api.main:app --reload --port 8000


from pathlib import Path
import joblib
import pandas as pd
import requests

# 1. Localizar a pasta principal do projeto
projeto = Path.cwd()

if projeto.name == "notebooks":
    projeto = projeto.parent

# 2. Carregar o Random Forest já treinado
caminho_modelo = projeto / "models" / "random_forest.joblib"

modelo = joblib.load(caminho_modelo)

print("Modelo carregado com sucesso!")

# 3. Preparar um registro real do dataset de teste
caminho_teste = projeto / "data" / "processed" / "dataset_teste.csv"

df_teste = pd.read_csv(caminho_teste)

colunas = [f"X{i}" for i in range(1, 24)]

dados_cliente = df_teste.loc[[0], colunas]

cliente_teste = dados_cliente.iloc[0].to_dict()

# 4. Calcular a previsão diretamente pelo modelo
probabilidade_notebook = float(
    modelo.predict_proba(dados_cliente)[0, 1]
)

# 5. Consultar a FastAPI
url = "http://127.0.0.1:8000/predict"

resposta = requests.post(
    url,
    json=cliente_teste,
    timeout=30
)

resposta.raise_for_status()

resultado_api = resposta.json()

probabilidade_api = resultado_api[
    "probabilidade_inadimplencia"
]

# 6. Comparar os resultados
diferenca = abs(
    probabilidade_notebook - probabilidade_api
)

print("\n=== VALIDAÇÃO FASTAPI VS NOTEBOOK ===")
print(f"HTTP: {resposta.status_code}")
print(f"Notebook: {probabilidade_notebook:.6f}")
print(f"FastAPI:  {probabilidade_api:.6f}")
print(f"Diferença: {diferenca:.10f}")

assert diferenca < 0.000001, (
    "Divergência entre API e notebook!"
)

print("\nSTATUS: VALIDAÇÃO APROVADA")


## Observação 4 — Validação do Random Forest via FastAPI

**Objetivo:** verificar se o modelo disponibilizado pela
FastAPI produz resultados equivalentes aos obtidos
diretamente no notebook Jupyter.

**Resultados obtidos:**

- HTTP: 200 — Sucesso
- Probabilidade no notebook: 0.136483
- Probabilidade na FastAPI: 0.136483
- Diferença absoluta: 0.0000000905
- Status: VALIDAÇÃO APROVADA

**Interpretação:**

O modelo Random Forest foi carregado a partir do
arquivo `models/random_forest.joblib`.

O mesmo registro do conjunto de teste foi processado
diretamente pelo modelo e pelo endpoint `/predict`.

Os resultados apresentaram equivalência dentro da
tolerância estabelecida de 0.000001.

A pequena diferença ocorreu devido ao arredondamento
da probabilidade retornada pela API.

**Conclusão:**

A integração entre o modelo de Machine Learning e a
FastAPI foi validada para o registro testado.

O endpoint `/predict` está preparado para os próximos
testes de integração com LangGraph, Ollama e n8n.